# 07 Naive Bayes

## Purpose

This notebook evaluates Naive Bayes classifiers for the Telco Customer Churn
project.

The previous learned-model sections introduced:

- logistic regression as a discriminative linear probability model;
- k-nearest neighbours as a non-parametric distance-based classifier.

Naive Bayes introduces a different modelling philosophy. It is a probabilistic
generative classifier: it models class priors and class-conditional feature
likelihoods, then uses Bayes' rule to form posterior class probabilities.

The deeper reusable theory is documented in:

```text
docs/knowledge_notes/models/07_naive_bayes.md
docs/knowledge_notes/methodology/evaluation_metrics.md
docs/knowledge_notes/methodology/hyperparameter_tuning.md
```

This notebook focuses on the executable workflow, model variants, cross-validated
outputs, and result inspection.

## Methodological discipline

The held-out test set is not used here.

All development-stage results are computed from stratified cross-validation
inside the training set.

Naive Bayes variants require model-specific preprocessing:

- GaussianNB is used with numeric features and continuous Gaussian likelihoods.
- BernoulliNB is used with one-hot encoded categorical features.
- A full transformed GaussianNB variant is included as a simple comparison, even
  though Gaussian likelihoods are not theoretically ideal for one-hot indicators.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

## Import project utilities

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    """Return the project root by searching upward for project marker files."""
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        has_project_markers = (
            (candidate / "pyproject.toml").exists()
            or (candidate / "README.md").exists()
        )
        has_project_dirs = (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
            and (candidate / "src").exists()
        )

        if has_project_markers and has_project_dirs:
            return candidate

    raise FileNotFoundError("Could not locate the project root directory.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [3]:
from telco_churn.config import (  # noqa: E402
    CATEGORICAL_FEATURES,
    FIGURES_DIR,
    NUMERIC_FEATURES,
    RANDOM_STATE,
    TABLES_DIR,
    TARGET_COLUMN,
)
from telco_churn.data import load_train_data, split_features_target  # noqa: E402
from telco_churn.evaluation import (  # noqa: E402
    evaluate_estimator_cv,
    evaluate_threshold_grid,
    get_out_of_fold_predictions,
    make_confusion_matrix_dataframe,
    make_precision_recall_curve_dataframe,
    make_roc_curve_dataframe,
    make_stratified_kfold,
)
from telco_churn.models import make_classifier_pipeline  # noqa: E402
from telco_churn.visualization import (  # noqa: E402
    save_precision_recall_curve_plot,
    save_roc_curve_plot,
    save_threshold_tradeoff_plot,
)

In [4]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:,.4f}".format)

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Output paths

In [5]:
NAIVE_BAYES_MODEL_COMPARISON_PATH = TABLES_DIR / "naive_bayes_model_comparison.csv"
NAIVE_BAYES_CONFUSION_MATRIX_PATH = TABLES_DIR / "naive_bayes_confusion_matrices.csv"
BERNOULLI_NB_ALPHA_RESULTS_PATH = TABLES_DIR / "bernoulli_nb_alpha_results.csv"
NAIVE_BAYES_THRESHOLD_RESULTS_PATH = TABLES_DIR / "naive_bayes_threshold_results.csv"

BERNOULLI_NB_ALPHA_FIGURE_PATH = FIGURES_DIR / "bernoulli_nb_alpha_metrics.png"
NAIVE_BAYES_THRESHOLD_FIGURE_PATH = FIGURES_DIR / "naive_bayes_threshold_tradeoff.png"
NAIVE_BAYES_ROC_CURVE_FIGURE_PATH = FIGURES_DIR / "naive_bayes_roc_curve.png"
NAIVE_BAYES_PRECISION_RECALL_CURVE_FIGURE_PATH = FIGURES_DIR / "naive_bayes_precision_recall_curve.png"

output_paths = pd.DataFrame(
    {
        "artifact": [
            "naive_bayes_model_comparison",
            "naive_bayes_confusion_matrices",
            "bernoulli_nb_alpha_results",
            "naive_bayes_threshold_results",
            "bernoulli_nb_alpha_figure",
            "naive_bayes_threshold_figure",
            "naive_bayes_roc_curve_figure",
            "naive_bayes_precision_recall_curve_figure",
        ],
        "path": [
            NAIVE_BAYES_MODEL_COMPARISON_PATH,
            NAIVE_BAYES_CONFUSION_MATRIX_PATH,
            BERNOULLI_NB_ALPHA_RESULTS_PATH,
            NAIVE_BAYES_THRESHOLD_RESULTS_PATH,
            BERNOULLI_NB_ALPHA_FIGURE_PATH,
            NAIVE_BAYES_THRESHOLD_FIGURE_PATH,
            NAIVE_BAYES_ROC_CURVE_FIGURE_PATH,
            NAIVE_BAYES_PRECISION_RECALL_CURVE_FIGURE_PATH,
        ],
    }
)

output_paths

,artifact,path
0,naive_bayes_model_comparison,C:\Projects_Data\classification\telco-customer...
1,naive_bayes_confusion_matrices,C:\Projects_Data\classification\telco-customer...
2,bernoulli_nb_alpha_results,C:\Projects_Data\classification\telco-customer...
3,naive_bayes_threshold_results,C:\Projects_Data\classification\telco-customer...
4,bernoulli_nb_alpha_figure,C:\Projects_Data\classification\telco-customer...
5,naive_bayes_threshold_figure,C:\Projects_Data\classification\telco-customer...
6,naive_bayes_roc_curve_figure,C:\Projects_Data\classification\telco-customer...
7,naive_bayes_precision_recall_curve_figure,C:\Projects_Data\classification\telco-customer...


## Load training data only

In [6]:
train_df = load_train_data()
X, y = split_features_target(train_df)

training_overview = pd.DataFrame(
    {
        "item": [
            "training_rows",
            "training_columns",
            "target_column",
            "positive_rate",
            "missing_values",
            "numeric_features",
            "categorical_features",
        ],
        "value": [
            train_df.shape[0],
            train_df.shape[1],
            TARGET_COLUMN,
            y.mean(),
            int(train_df.isna().sum().sum()),
            len(NUMERIC_FEATURES),
            len(CATEGORICAL_FEATURES),
        ],
    }
)

training_overview

,item,value
0,training_rows,5634
1,training_columns,20
2,target_column,Churn_binary
3,positive_rate,0.2654
4,missing_values,0
5,numeric_features,3
6,categorical_features,16


In [7]:
target_distribution = (
    y.value_counts(normalize=False)
    .rename("count")
    .to_frame()
    .assign(percentage=lambda df: 100 * df["count"] / df["count"].sum())
    .rename_axis(TARGET_COLUMN)
    .reset_index()
)

target_distribution

,Churn_binary,count,percentage
0,0,4139,73.4647
1,1,1495,26.5353


## Naive Bayes theory needed for this notebook

The ideal Bayes classifier predicts the class with the largest true posterior
probability:

$$
h^\star(x)
=
\arg\max_{y \in \{0,1\}}
P(Y=y \mid X=x).
$$

The true posterior is unknown. Naive Bayes approximates it by estimating the
class prior and class-conditional likelihoods:

$$
P(Y=y),
\qquad
P(X=x \mid Y=y).
$$

The "naive" assumption is conditional independence:

$$
P(X=x \mid Y=y)
=
\prod_{j=1}^{p}
P(X_j=x_j \mid Y=y).
$$

This assumption is not literally true for the Telco features, but it can still
produce useful classification and ranking performance.

## Cross-validation

In [8]:
cv = make_stratified_kfold()

cv_check = pd.DataFrame(
    {
        "item": ["strategy", "n_splits", "shuffle", "random_state"],
        "value": ["StratifiedKFold", cv.n_splits, True, RANDOM_STATE],
    }
)

cv_check

,item,value
0,strategy,StratifiedKFold
1,n_splits,5
2,shuffle,True
3,random_state,42


## Helper functions for Naive Bayes preprocessing

Different Naive Bayes variants need different feature representations.

GaussianNB expects dense continuous features. BernoulliNB works naturally with
binary indicator features, so it is suitable for one-hot encoded categorical
variables.

In [9]:
def make_one_hot_encoder_for_nb(*, sparse_output: bool):
    """Create a one-hot encoder compatible with old and new scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=sparse_output)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=sparse_output)


def to_dense_array(X):
    """Convert sparse matrices to dense arrays for estimators that require dense input."""
    if sparse.issparse(X):
        return X.toarray()
    return X


def make_numeric_only_preprocessor() -> ColumnTransformer:
    """Create preprocessing for numeric-only GaussianNB."""
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ],
        remainder="drop",
    )


def make_categorical_onehot_preprocessor(*, sparse_output: bool = True) -> ColumnTransformer:
    """Create one-hot preprocessing for categorical-only BernoulliNB."""
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder_for_nb(sparse_output=sparse_output)),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
    )


def make_full_dense_onehot_preprocessor() -> Pipeline:
    """Create dense full-feature preprocessing for GaussianNB.

    Numeric features are median-imputed and categorical features are one-hot
    encoded. The combined matrix is converted to dense form because GaussianNB
    does not accept sparse matrices.
    """
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder_for_nb(sparse_output=True)),
        ]
    )

    column_transformer = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, NUMERIC_FEATURES),
            ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
    )

    return Pipeline(
        steps=[
            ("columns", column_transformer),
            ("dense", FunctionTransformer(to_dense_array, accept_sparse=True)),
        ]
    )


def make_gaussian_numeric_nb_pipeline() -> Pipeline:
    """Create numeric-only GaussianNB pipeline."""
    return make_classifier_pipeline(
        preprocessor=make_numeric_only_preprocessor(),
        classifier=GaussianNB(),
    )


def make_bernoulli_categorical_nb_pipeline(*, alpha: float = 1.0) -> Pipeline:
    """Create categorical-only BernoulliNB pipeline."""
    return make_classifier_pipeline(
        preprocessor=make_categorical_onehot_preprocessor(sparse_output=True),
        classifier=BernoulliNB(alpha=alpha),
    )


def make_gaussian_full_nb_pipeline() -> Pipeline:
    """Create full transformed GaussianNB pipeline."""
    return make_classifier_pipeline(
        preprocessor=make_full_dense_onehot_preprocessor(),
        classifier=GaussianNB(),
    )


def save_alpha_metric_plot(
    *,
    results_df: pd.DataFrame,
    output_path: Path,
    title: str,
    metric_columns: list[str] | None = None,
) -> None:
    """Save BernoulliNB smoothing metrics over alpha."""
    if metric_columns is None:
        metric_columns = ["pr_auc", "roc_auc", "balanced_accuracy", "f1"]

    fig, ax = plt.subplots(figsize=(9, 5.5))

    plot_df = results_df.sort_values("alpha")

    for metric in metric_columns:
        ax.plot(plot_df["alpha"], plot_df[metric], marker="o", label=metric)

    ax.set_xscale("log")
    ax.set_xlabel("Additive smoothing alpha")
    ax.set_ylabel("Cross-validated metric")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

## Naive Bayes model variants

We evaluate three transparent variants:

1. **GaussianNB numeric only**: uses the three numeric features.
2. **BernoulliNB categorical only**: uses one-hot encoded categorical features.
3. **GaussianNB full transformed**: uses numeric features plus one-hot encoded
   categorical indicators as a simple full-feature GaussianNB comparison.

The third variant is not the cleanest theoretical likelihood for one-hot
indicators, but it is useful as a simple full-feature benchmark.

In [10]:
nb_estimators = {
    "GaussianNB numeric only": make_gaussian_numeric_nb_pipeline(),
    "BernoulliNB categorical only alpha=1": make_bernoulli_categorical_nb_pipeline(alpha=1.0),
    "GaussianNB full transformed": make_gaussian_full_nb_pipeline(),
}

nb_results = []

for model_name, estimator in nb_estimators.items():
    result = evaluate_estimator_cv(
        model_name=model_name,
        estimator=estimator,
        X=X,
        y=y,
        cv=cv,
    )
    nb_results.append(result)

nb_results_df = pd.DataFrame(nb_results)

nb_metric_columns = [
    "model",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "roc_auc",
    "pr_auc",
    "predicted_positive_rate",
    "observed_positive_rate",
]

nb_metric_df = (
    nb_results_df[nb_metric_columns]
    .sort_values(["pr_auc", "balanced_accuracy", "f1"], ascending=False)
    .reset_index(drop=True)
)

nb_metric_df

,model,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,pr_auc,predicted_positive_rate,observed_positive_rate
0,GaussianNB full transformed,0.6977,0.7456,0.4621,0.8475,0.6436,0.5981,0.8192,0.6049,0.4867,0.2654
1,BernoulliNB categorical only alpha=1,0.7210,0.7507,0.4847,0.8140,0.6874,0.6076,0.8150,0.5955,0.4457,0.2654
2,GaussianNB numeric only,0.7645,0.6957,0.5570,0.5492,0.8422,0.5530,0.7744,0.5838,0.2616,0.2654


In [11]:
nb_confusion_df = (
    make_confusion_matrix_dataframe(nb_results_df)
    .set_index("model")
    .loc[nb_metric_df["model"]]
    .reset_index()
)

nb_confusion_df

,model,tp,fn,fp,tn
0,GaussianNB full transformed,1267,228,1475,2664
1,BernoulliNB categorical only alpha=1,1217,278,1294,2845
2,GaussianNB numeric only,821,674,653,3486


## BernoulliNB smoothing grid

BernoulliNB uses additive smoothing. Smoothing prevents zero probabilities and
controls how strongly rare indicator patterns affect the likelihood.

We tune alpha only for the categorical-only BernoulliNB model.

In [12]:
ALPHA_GRID = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

alpha_results = []

for alpha in ALPHA_GRID:
    estimator = make_bernoulli_categorical_nb_pipeline(alpha=alpha)
    result = evaluate_estimator_cv(
        model_name=f"BernoulliNB categorical only alpha={alpha}",
        estimator=estimator,
        X=X,
        y=y,
        cv=cv,
    )
    result["alpha"] = alpha
    alpha_results.append(result)

bernoulli_alpha_results_df = pd.DataFrame(alpha_results).sort_values("alpha").reset_index(drop=True)

alpha_display_columns = [
    "alpha",
    "accuracy",
    "balanced_accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "roc_auc",
    "pr_auc",
    "predicted_positive_rate",
]

bernoulli_alpha_results_df[alpha_display_columns]

,alpha,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,pr_auc,predicted_positive_rate
0,0.0100,0.7213,0.7509,0.4851,0.8140,0.6878,0.6079,0.8151,0.5956,0.4453
1,0.0500,0.7213,0.7509,0.4851,0.8140,0.6878,0.6079,0.8151,0.5956,0.4453
2,0.1000,0.7213,0.7509,0.4851,0.8140,0.6878,0.6079,0.8151,0.5956,0.4453
3,0.5000,0.7212,0.7508,0.4849,0.8140,0.6876,0.6077,0.8151,0.5956,0.4455
4,1.0000,0.7210,0.7507,0.4847,0.8140,0.6874,0.6076,0.8150,0.5955,0.4457
5,2.0000,0.7212,0.7510,0.4849,0.8147,0.6874,0.6079,0.8150,0.5955,0.4459
6,5.0000,0.7215,0.7519,0.4853,0.8167,0.6871,0.6088,0.8150,0.5955,0.4466
7,10.0000,0.7208,0.7514,0.4845,0.8167,0.6862,0.6082,0.8149,0.5954,0.4473


## Select representative Naive Bayes model

The representative Naive Bayes model is selected by cross-validated PR-AUC, with
balanced accuracy and F1 as secondary tie-breakers. This is the same selection
logic used in the kNN section.

In [13]:
candidate_rows = []

# Main untuned variants.
for _, row in nb_results_df.iterrows():
    candidate = row.to_dict()
    candidate["variant_source"] = "main_variant"
    candidate["alpha"] = np.nan
    candidate_rows.append(candidate)

# Bernoulli smoothing variants.
for _, row in bernoulli_alpha_results_df.iterrows():
    candidate = row.to_dict()
    candidate["variant_source"] = "bernoulli_alpha_grid"
    candidate_rows.append(candidate)

nb_candidate_results_df = pd.DataFrame(candidate_rows)

best_nb_row = nb_candidate_results_df.sort_values(
    ["pr_auc", "balanced_accuracy", "f1"],
    ascending=False,
).iloc[0]

selection_summary = pd.DataFrame(
    {
        "item": [
            "selection_rule",
            "selected_model",
            "selected_variant_source",
            "selected_alpha",
            "selected_pr_auc",
            "selected_roc_auc",
            "selected_balanced_accuracy",
            "selected_f1",
        ],
        "value": [
            "highest cross-validated PR-AUC among Naive Bayes candidates",
            best_nb_row["model"],
            best_nb_row["variant_source"],
            best_nb_row["alpha"],
            best_nb_row["pr_auc"],
            best_nb_row["roc_auc"],
            best_nb_row["balanced_accuracy"],
            best_nb_row["f1"],
        ],
    }
)

selection_summary

,item,value
0,selection_rule,highest cross-validated PR-AUC among Naive Bay...
1,selected_model,GaussianNB full transformed
2,selected_variant_source,main_variant
3,selected_alpha,NaN
4,selected_pr_auc,0.6049
5,selected_roc_auc,0.8192
6,selected_balanced_accuracy,0.7456
7,selected_f1,0.5981


## Recreate selected Naive Bayes pipeline

In [14]:
selected_model_name = str(best_nb_row["model"])

if selected_model_name.startswith("GaussianNB numeric only"):
    selected_nb_pipeline = make_gaussian_numeric_nb_pipeline()
elif selected_model_name.startswith("GaussianNB full transformed"):
    selected_nb_pipeline = make_gaussian_full_nb_pipeline()
elif selected_model_name.startswith("BernoulliNB categorical only"):
    selected_alpha = float(best_nb_row["alpha"])
    if np.isnan(selected_alpha):
        selected_alpha = 1.0
    selected_nb_pipeline = make_bernoulli_categorical_nb_pipeline(alpha=selected_alpha)
else:
    raise ValueError(f"Unknown selected model: {selected_model_name}")

selected_nb_result = evaluate_estimator_cv(
    model_name=f"Selected {selected_model_name}",
    estimator=selected_nb_pipeline,
    X=X,
    y=y,
    cv=cv,
)

selected_nb_result_df = pd.DataFrame([selected_nb_result])
selected_nb_result_df[nb_metric_columns]

,model,accuracy,balanced_accuracy,precision,recall,specificity,f1,roc_auc,pr_auc,predicted_positive_rate,observed_positive_rate
0,Selected GaussianNB full transformed,0.6977,0.7456,0.4621,0.8475,0.6436,0.5981,0.8192,0.6049,0.4867,0.2654


## Threshold tradeoff for selected Naive Bayes

The selected Naive Bayes model produces predicted probabilities. Because Naive
Bayes can be overconfident when features are correlated, threshold behaviour is
important. The threshold analysis uses out-of-fold predicted probabilities.

In [15]:
_, selected_nb_oof_probability = get_out_of_fold_predictions(
    estimator=selected_nb_pipeline,
    X=X,
    y=y,
    cv=cv,
)

thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)

nb_threshold_df = evaluate_threshold_grid(
    y_true=y,
    y_score=selected_nb_oof_probability,
    thresholds=thresholds,
)

nb_threshold_df

,threshold,tp,fn,fp,tn,accuracy,balanced_accuracy,precision,recall,specificity,f1,predicted_positive_rate,observed_positive_rate
0,0.0500,1306,189,1661,2478,0.6716,0.7361,0.4402,0.8736,0.5987,0.5854,0.5266,0.2654
1,0.1000,1294,201,1617,2522,0.6773,0.7374,0.4445,0.8656,0.6093,0.5874,0.5167,0.2654
2,0.1500,1285,210,1590,2549,0.6805,0.7377,0.4470,0.8595,0.6158,0.5881,0.5103,0.2654
3,0.2000,1281,214,1562,2577,0.6848,0.7397,0.4506,0.8569,0.6226,0.5906,0.5046,0.2654
4,0.2500,1279,216,1550,2589,0.6865,0.7405,0.4521,0.8555,0.6255,0.5916,0.5021,0.2654
5,0.3000,1276,219,1529,2610,0.6897,0.7420,0.4549,0.8535,0.6306,0.5935,0.4979,0.2654
6,0.3500,1273,222,1517,2622,0.6913,0.7425,0.4563,0.8515,0.6335,0.5942,0.4952,0.2654
7,0.4000,1273,222,1505,2634,0.6935,0.7439,0.4582,0.8515,0.6364,0.5958,0.4931,0.2654
8,0.4500,1271,224,1490,2649,0.6958,0.7451,0.4603,0.8502,0.6400,0.5973,0.4901,0.2654
9,0.5000,1267,228,1475,2664,0.6977,0.7456,0.4621,0.8475,0.6436,0.5981,0.4867,0.2654


## ROC and precision-recall curves for selected Naive Bayes

In [16]:
nb_roc_curve_df = make_roc_curve_dataframe(
    y_true=y,
    y_score=selected_nb_oof_probability,
)

nb_precision_recall_curve_df = make_precision_recall_curve_dataframe(
    y_true=y,
    y_score=selected_nb_oof_probability,
)

curve_summary = pd.DataFrame(
    {
        "curve": ["ROC", "Precision-recall"],
        "rows": [len(nb_roc_curve_df), len(nb_precision_recall_curve_df)],
        "baseline_reference": [
            "diagonal random-ranking line",
            f"positive-rate baseline = {y.mean():.4f}",
        ],
    }
)

curve_summary

,curve,rows,baseline_reference
0,ROC,1594,diagonal random-ranking line
1,Precision-recall,5626,positive-rate baseline = 0.2654


## Save tables and figures

In [17]:
nb_metric_df.to_csv(NAIVE_BAYES_MODEL_COMPARISON_PATH, index=False)
nb_confusion_df.to_csv(NAIVE_BAYES_CONFUSION_MATRIX_PATH, index=False)
bernoulli_alpha_results_df.to_csv(BERNOULLI_NB_ALPHA_RESULTS_PATH, index=False)
nb_threshold_df.to_csv(NAIVE_BAYES_THRESHOLD_RESULTS_PATH, index=False)

save_alpha_metric_plot(
    results_df=bernoulli_alpha_results_df,
    output_path=BERNOULLI_NB_ALPHA_FIGURE_PATH,
    title="Bernoulli Naive Bayes smoothing metrics",
)

nb_threshold_plot_df = nb_threshold_df[nb_threshold_df["threshold"] <= 0.80].copy()

save_threshold_tradeoff_plot(
    threshold_df=nb_threshold_plot_df,
    output_path=NAIVE_BAYES_THRESHOLD_FIGURE_PATH,
    title="Selected Naive Bayes Threshold Tradeoff",
)

save_roc_curve_plot(
    roc_curve_df=nb_roc_curve_df,
    output_path=NAIVE_BAYES_ROC_CURVE_FIGURE_PATH,
    title="Selected Naive Bayes ROC Curve",
)

save_precision_recall_curve_plot(
    precision_recall_curve_df=nb_precision_recall_curve_df,
    output_path=NAIVE_BAYES_PRECISION_RECALL_CURVE_FIGURE_PATH,
    title="Selected Naive Bayes Precision-Recall Curve",
    positive_rate=float(y.mean()),
)

saved_artifacts = pd.DataFrame(
    {
        "artifact": [
            "naive_bayes_model_comparison",
            "naive_bayes_confusion_matrices",
            "bernoulli_nb_alpha_results",
            "naive_bayes_threshold_results",
            "bernoulli_nb_alpha_figure",
            "naive_bayes_threshold_figure",
            "naive_bayes_roc_curve_figure",
            "naive_bayes_precision_recall_curve_figure",
        ],
        "exists": [
            NAIVE_BAYES_MODEL_COMPARISON_PATH.exists(),
            NAIVE_BAYES_CONFUSION_MATRIX_PATH.exists(),
            BERNOULLI_NB_ALPHA_RESULTS_PATH.exists(),
            NAIVE_BAYES_THRESHOLD_RESULTS_PATH.exists(),
            BERNOULLI_NB_ALPHA_FIGURE_PATH.exists(),
            NAIVE_BAYES_THRESHOLD_FIGURE_PATH.exists(),
            NAIVE_BAYES_ROC_CURVE_FIGURE_PATH.exists(),
            NAIVE_BAYES_PRECISION_RECALL_CURVE_FIGURE_PATH.exists(),
        ],
        "path": [
            NAIVE_BAYES_MODEL_COMPARISON_PATH,
            NAIVE_BAYES_CONFUSION_MATRIX_PATH,
            BERNOULLI_NB_ALPHA_RESULTS_PATH,
            NAIVE_BAYES_THRESHOLD_RESULTS_PATH,
            BERNOULLI_NB_ALPHA_FIGURE_PATH,
            NAIVE_BAYES_THRESHOLD_FIGURE_PATH,
            NAIVE_BAYES_ROC_CURVE_FIGURE_PATH,
            NAIVE_BAYES_PRECISION_RECALL_CURVE_FIGURE_PATH,
        ],
    }
)

saved_artifacts

,artifact,exists,path
0,naive_bayes_model_comparison,True,C:\Projects_Data\classification\telco-customer...
1,naive_bayes_confusion_matrices,True,C:\Projects_Data\classification\telco-customer...
2,bernoulli_nb_alpha_results,True,C:\Projects_Data\classification\telco-customer...
3,naive_bayes_threshold_results,True,C:\Projects_Data\classification\telco-customer...
4,bernoulli_nb_alpha_figure,True,C:\Projects_Data\classification\telco-customer...
5,naive_bayes_threshold_figure,True,C:\Projects_Data\classification\telco-customer...
6,naive_bayes_roc_curve_figure,True,C:\Projects_Data\classification\telco-customer...
7,naive_bayes_precision_recall_curve_figure,True,C:\Projects_Data\classification\telco-customer...


## Summary before result interpretation

This notebook has now produced the first Naive Bayes results.

After running the notebook, inspect:

```text
nb_metric_df
nb_confusion_df
bernoulli_alpha_results_df
selection_summary
selected_nb_result_df
nb_threshold_df
saved figures
```

The next step is to interpret the actual outputs and then update this notebook
with concise result interpretation before writing the LaTeX report section.